In [2]:
import numpy as np
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df = pd.read_pickle('telemetry_9.pkl')
df = df.loc[10:200,:].reset_index()


In [3]:
from scipy.signal import find_peaks
from scipy.signal import savgol_filter

df = pd.read_pickle('telemetry_71.pkl')
df = df.loc[10:200,:].reset_index()

modes = ["idle","accel","cruising","decel","coasting","overrun","tip-in","tip-out","WOT"]
df["Mode"] = "DECEL"

df['Throttle2'] = df['Throttle'].rolling(window=10, center=True).mean()


df.loc[df['Throttle']>=100,"Mode"] = "WOT"
df.loc[(df['Throttle']>0) & (df['Throttle']<100),"Mode"] = "ACCEL"


peaks, _ = find_peaks(df['Throttle'], height=0, prominence=0)  # Specify parameters like height if needed

df['dThrottle'] = df['Throttle'].diff()
df.loc[df['dThrottle'] < 0, 'Mode'] = 'DECEL'


fig = make_subplots(rows=4, cols=1, shared_xaxes=True)
fig.add_trace(go.Scatter(x=df['Time'],y=df['Throttle'],mode='lines',yaxis="y",name="Throttle"),row=1, col=1)
fig.add_trace(go.Scatter(x=df.loc[peaks,'Time'],y=df.loc[peaks,'Throttle'],mode='markers',yaxis="y",name="Peaks"),row=1, col=1)


fig.add_trace(go.Scatter(x=df['Time'],y=df['Brake'],mode='lines',yaxis="y",name="Brake"),row=2, col=1)
fig.add_trace(go.Scatter(x=df['Time'],y=df['Speed'],mode='lines',yaxis="y",name="Speed"),row=3, col=1)
fig.add_trace(go.Scatter(x=df['Time'],y=df['Mode'],mode='lines',yaxis="y",name="Mode"),row=4, col=1)
fig.update_layout(margin=dict(l=20, r=20, t=20, b=20))
fig.update_yaxes(autorange="reversed",row=4, col=1)
fig.show()

In [6]:
from sklearn.linear_model import LinearRegression

df['Event'] = (df['Mode'] != df['Mode'].shift()).cumsum()

# Group by events and aggregate
result = df.groupby('Event').agg({
    'Time': ['min', 'max'],               # Start and end times of each mode
    'Mode': 'first',              # Driving mode for the event
}).reset_index(drop=True)
result.columns = ['Start Time', 'End Time', 'Driving Mode']


start_indices = []
end_indices = []
grad = []

for _, row_a in result.iterrows():
    start_time = row_a['Start Time']
    end_time = row_a['End Time']

    # Find the indices in DataFrame B where the time falls within the range
    start_index = df[df['Time'] >= start_time].index.min()
    end_index = df[df['Time'] <= end_time].index.max() + 1

    X = df.loc[start_index:end_index,'Speed'].values
    y = df.loc[start_index:end_index,'Time'].values

    model = LinearRegression()
    model.fit(X, y)
    overall_gradient = model.coef_[0]  # The slope of the line

    # Append the indices to the respective lists
    start_indices.append(start_index)
    end_indices.append(end_index)
    grad.append(overall_gradient)

result['Start Index'] = start_indices
result['End Index'] = end_indices
result['Speed Gradient'] = grad


result


ValueError: Expected 2D array, got 1D array instead:
array=[217 214 202 191 180 172 165 154 143 141 139 133 127 121 115 111 107 104
 101  97  94  92  90  88  87].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [155]:
import json

presult = result[["Start Time","End Time","Driving Mode"]]
presult
json_dict = presult.to_dict(orient='records')
json_dict

json_string = json.dumps(json_dict)
json_string

'[{"Start Time": 1.357, "End Time": 3.075, "Driving Mode": "BRAKE"}, {"Start Time": 4.157, "End Time": 4.855, "Driving Mode": "ACCEL"}, {"Start Time": 5.037, "End Time": 10.637, "Driving Mode": "WOT"}, {"Start Time": 10.715, "End Time": 10.715, "Driving Mode": "DECEL"}, {"Start Time": 10.837, "End Time": 12.795, "Driving Mode": "BRAKE"}, {"Start Time": 13.037, "End Time": 14.717, "Driving Mode": "ACCEL"}, {"Start Time": 14.815, "End Time": 15.555, "Driving Mode": "DECEL"}, {"Start Time": 15.637, "End Time": 16.235, "Driving Mode": "BRAKE"}, {"Start Time": 16.397, "End Time": 18.255, "Driving Mode": "ACCEL"}, {"Start Time": 18.277, "End Time": 19.277, "Driving Mode": "WOT"}, {"Start Time": 19.437, "End Time": 19.495, "Driving Mode": "DECEL"}, {"Start Time": 19.757, "End Time": 22.557, "Driving Mode": "BRAKE"}, {"Start Time": 22.755, "End Time": 24.717, "Driving Mode": "ACCEL"}, {"Start Time": 24.917, "End Time": 25.015, "Driving Mode": "DECEL"}, {"Start Time": 25.077, "End Time": 27.837